# Arabic Letter and Number Pronunciation Experiments using Edge TTS

This notebook documents the **pronunciation experimentation and manual selection** workflow for Egyptian Arabic letters (28) and numbers (0–10).

- Multiple pronunciation variants were generated with **Edge TTS** (`ar-EG-SalmaNeural`).
- Edge TTS output can differ slightly between runs, so **final approved MP3s are not regenerated here**.
- After listening tests, one variant per label was promoted to `generated_audio/.../chosen/`.
- The **real-time prediction pipeline** loads the exported files from `audio/letters/` and `audio/numbers/` directly (no live TTS at inference time).


## Imports and Paths

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)

_ensure("pandas")
_ensure("IPython", "IPython")

import pandas as pd
from IPython.display import Audio, Markdown, display

BASE_DIR = Path.cwd().resolve()
GEN = BASE_DIR / "generated_audio"

DIR_LETTERS_CHOSEN = GEN / "edge_tts_mapping_v2" / "chosen"
DIR_LETTERS_COMP = GEN / "edge_tts_mapping_v2" / "comparison"
DIR_LETTERS_COMP_CUT = DIR_LETTERS_COMP / "cut"

DIR_NUMBERS_CHOSEN = GEN / "edge_tts_numbers_mapping_v1" / "chosen"
DIR_NUMBERS_COMP = GEN / "edge_tts_numbers_mapping_v1" / "comparison"

DIR_EXPORT_LETTERS = BASE_DIR / "audio" / "letters"
DIR_EXPORT_NUMBERS = BASE_DIR / "audio" / "numbers"

for d in (DIR_EXPORT_LETTERS, DIR_EXPORT_NUMBERS):
    d.mkdir(parents=True, exist_ok=True)

LETTER_ORDER = list("ابتثجحخدذرزسشصضطظعغفقكلمنهوي")

print("cwd:", BASE_DIR)
print("letters chosen:", DIR_LETTERS_CHOSEN)
print("letters comparison:", DIR_LETTERS_COMP)
print("numbers chosen:", DIR_NUMBERS_CHOSEN)
print("export letters:", DIR_EXPORT_LETTERS)
print("export numbers:", DIR_EXPORT_NUMBERS)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff
letters chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen
letters comparison: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison
numbers chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers_mapping_v1\chosen
export letters: C:\Users\sondo\Desktop\Voice Model Stuff\audio\letters
export numbers: C:\Users\sondo\Desktop\Voice Model Stuff\audio\numbers


## Edge TTS Voice Setup

Voice used during experimentation: **`ar-EG-SalmaNeural`** (fallback `ar-EG-ShakirNeural`).

This section only records the voice choice. **No synthesis runs in this notebook.**


In [2]:
# Document voice used in earlier edge notebooks (no network synthesis here).
EDGE_VOICE_USED = "ar-EG-SalmaNeural"
EDGE_VOICE_FALLBACK = "ar-EG-ShakirNeural"
print("Documented Edge TTS voice:", EDGE_VOICE_USED)
print("Fallback:", EDGE_VOICE_FALLBACK)


Documented Edge TTS voice: ar-EG-SalmaNeural
Fallback: ar-EG-ShakirNeural


## Letter Pronunciation Variants

During experimentation we tested several strategies when the default letter name sounded wrong:

| Strategy | Examples | Why |
|----------|----------|-----|
| **Sukoon ending** | `كافْ.`, `صَاادْ.`, `ضَاادْ.` | Stops unwanted vowel carry |
| **Long vowel spelling** | `صَاادْ`, `ضَاادْ` | Emphasizes elongated articulation |
| **Latin transliteration** | `lam.`, `zeen.` | Sometimes clearer for Edge TTS |
| **Bang / exclamation** | `دال!`, `ذال!` | Short, punchy Egyptian-style stop |
| **Spaced syllables + cut** | `س ي ن.` → 0.6s clip; `قا فْ.` → two-stage cut | Controls pacing and trailing noise |
| **Diacritic spelling** | `وَاْو.` | Fine-tunes واو pronunciation |

Variants live under `generated_audio/edge_tts_mapping_v2/comparison/` (and `comparison/cut/`). Winners were copied or moved into `chosen/`.


In [3]:
def _letter_from_stem(stem: str) -> str | None:
    for ch in LETTER_ORDER:
        if f"_{ch}_" in stem or stem.endswith(f"_{ch}"):
            return ch
    return None

def list_mp3s(folder: Path) -> list[Path]:
    if not folder.is_dir():
        return []
    return sorted(folder.glob("*.mp3"))

comp_files = list_mp3s(DIR_LETTERS_COMP)
cut_files = list_mp3s(DIR_LETTERS_COMP_CUT)
all_variants = comp_files + cut_files

rows = []
for p in all_variants:
    stem = p.stem
    rows.append({
        "letter": _letter_from_stem(stem) or "?",
        "variant_file": p.name,
        "folder": p.parent.name,
        "in_chosen": (DIR_LETTERS_CHOSEN / p.name).is_file(),
    })

variant_df = pd.DataFrame(rows)
print(f"Letter comparison variants on disk: {len(variant_df)}")
if len(variant_df):
    display(variant_df.sort_values(["letter", "variant_file"]).head(40))
    if len(variant_df) > 40:
        print(f"... and {len(variant_df) - 40} more")
else:
    print("(No comparison folder files — variants may have been archived.)")


Letter comparison variants on disk: 86


,letter,variant_file,folder,in_chosen
0,ا,comp_beh_ب_ب_ا.mp3,comparison,False
1,ب,comp_beh_ب_ب_ه.mp3,comparison,False
2,ب,comp_beh_ب_ب_ي.mp3,comparison,False
3,ب,comp_beh_ب_ب_ي_ه.mp3,comparison,False
15,د,comp_dal_د_daal_latin_long.mp3,comparison,False
16,د,comp_dal_د_دآل_madd.mp3,comparison,False
17,د,comp_dal_د_دال_diac.mp3,comparison,False
18,د,comp_dal_د_دال_plain.mp3,comparison,False
19,د,comp_dal_د_دال_sukoon.mp3,comparison,False
72,ذ,comp_zal_ذ_dhaal_latin_long.mp3,comparison,False


... and 46 more


## Number Pronunciation Variants

Numbers 0–10 were tested in `edge_tts_numbers_mapping_v1/comparison/` (e.g. multiple stems for **0** and **2**, and **10** with sukoon).

Notable selections:
- **0**: `num_00_v02` (variant v02 of صِفِر)
- **2**: `num_02_atneen_bang` with **0.59s** leading-trim cut (`num_02_atneen_bang_cut_0_59`)
- **10**: `num_10_asharah_sukoon` (عشرةْ)


In [4]:
num_comp = list_mp3s(DIR_NUMBERS_COMP)
num_chosen = list_mp3s(DIR_NUMBERS_CHOSEN)

num_rows = []
for p in sorted(set(num_comp + num_chosen)):
    stem = p.stem
    label = stem.split("_")[1] if stem.startswith("num_") else stem
    num_rows.append({
        "stem": stem,
        "file": p.name,
        "in_chosen": p.parent == DIR_NUMBERS_CHOSEN,
        "in_comparison": p.parent == DIR_NUMBERS_COMP,
    })

num_variant_df = pd.DataFrame(num_rows)
print(f"Number files (chosen + comparison): {len(num_variant_df)}")
display(num_variant_df.sort_values("stem"))


Number files (chosen + comparison): 12


,stem,file,in_chosen,in_comparison
0,num_00_v02,num_00_v02.mp3,True,False
1,num_01,num_01.mp3,True,False
11,num_02_atneen_bang,num_02_atneen_bang.mp3,False,True
2,num_02_atneen_bang_cut_0_59,num_02_atneen_bang_cut_0_59.mp3,True,False
3,num_03,num_03.mp3,True,False
4,num_04,num_04.mp3,True,False
5,num_05,num_05.mp3,True,False
6,num_06,num_06.mp3,True,False
7,num_07,num_07.mp3,True,False
8,num_08,num_08.mp3,True,False


## Approved Final Selections

Curated **chosen** MP3s (already generated — **not re-synthesized**). Each row maps a label to the approved source file and export name.

Cut flags describe whether the approved file is itself a trimmed clip from experimentation.


In [5]:
APPROVED_LETTER_EXPORT = [
    {"label": "ا", "approved_filename": "letter_ا_alif.mp3", "pronunciation_style": "standard (أَلِف)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ب", "approved_filename": "letter_ب_cut_0.45.mp3", "pronunciation_style": "بِه + 0.45s duration cut", "cut_applied": True, "selection_notes": "Trimmed beh clip; avoids long tail"},
    {"label": "ت", "approved_filename": "letter_ت_teh.mp3", "pronunciation_style": "standard (تِه)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ث", "approved_filename": "comp_seh_ث_ث_ه.mp3", "pronunciation_style": "ثِه (comp)", "cut_applied": False, "selection_notes": "Comparison winner"},
    {"label": "ج", "approved_filename": "letter_ج_geem.mp3", "pronunciation_style": "standard (جِيم)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ح", "approved_filename": "letter_ح_ha.mp3", "pronunciation_style": "standard (حَا)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "خ", "approved_filename": "letter_خ_kha.mp3", "pronunciation_style": "standard (خَه)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "د", "approved_filename": "comp_dal_د_دال_bang.mp3", "pronunciation_style": "دال! (bang)", "cut_applied": False, "selection_notes": "Comparison winner over latin variant"},
    {"label": "ذ", "approved_filename": "comp_zal_ذ_ذال_bang.mp3", "pronunciation_style": "ذال! (bang)", "cut_applied": False, "selection_notes": "Comparison winner"},
    {"label": "ر", "approved_filename": "letter_ر_ra.mp3", "pronunciation_style": "standard (رَا)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ز", "approved_filename": "comp_zay_ز_zeen_latin_long.mp3", "pronunciation_style": "Latin zeen.", "cut_applied": False, "selection_notes": "Comparison winner"},
    {"label": "س", "approved_filename": "comp_seen_س_seen_split_cut.mp3", "pronunciation_style": "س ي ن + cut", "cut_applied": True, "selection_notes": "Spaced syllables, 0.6s clip"},
    {"label": "ش", "approved_filename": "letter_ش_sheen.mp3", "pronunciation_style": "standard (شِين)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ص", "approved_filename": "comp_sad_ص_saad_long_sukoon.mp3", "pronunciation_style": "long + sukoon (صَاادْ)", "cut_applied": False, "selection_notes": "Round-6 comparison winner"},
    {"label": "ض", "approved_filename": "comp_dad_ض_daad_long_sukoon.mp3", "pronunciation_style": "long + sukoon (ضَاادْ)", "cut_applied": False, "selection_notes": "Round-6 comparison winner"},
    {"label": "ط", "approved_filename": "comp_ta_ط_ط_ه.mp3", "pronunciation_style": "طَه (comp)", "cut_applied": False, "selection_notes": "Comparison mapping"},
    {"label": "ظ", "approved_filename": "letter_ظ_dha.mp3", "pronunciation_style": "standard (ظَه)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ع", "approved_filename": "letter_ع_ein.mp3", "pronunciation_style": "standard (عِين)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "غ", "approved_filename": "letter_غ_ghein.mp3", "pronunciation_style": "standard (غِين)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ف", "approved_filename": "comp_faa_ف_ف_ه.mp3", "pronunciation_style": "فِه (comp)", "cut_applied": False, "selection_notes": "Comparison mapping"},
    {"label": "ق", "approved_filename": "comp_qaf_ق_qaf_split_cut_v2.mp3", "pronunciation_style": "قا فْ + two-stage cut", "cut_applied": True, "selection_notes": "v2 cut kept first 0.61s after trailing trim"},
    {"label": "ك", "approved_filename": "comp_kaf_ك_كاف_sukoon.mp3", "pronunciation_style": "كافْ (sukoon)", "cut_applied": False, "selection_notes": "Comparison winner"},
    {"label": "ل", "approved_filename": "comp_lam_ل_lam_latin.mp3", "pronunciation_style": "Latin lam.", "cut_applied": False, "selection_notes": "Comparison winner"},
    {"label": "م", "approved_filename": "letter_م_meem.mp3", "pronunciation_style": "standard (مِيم)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ن", "approved_filename": "letter_ن_noon.mp3", "pronunciation_style": "standard (نُون)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
    {"label": "ه", "approved_filename": "comp_ha_ه_ه_ه.mp3", "pronunciation_style": "هِه (comp)", "cut_applied": False, "selection_notes": "Comparison mapping"},
    {"label": "و", "approved_filename": "comp_waw_و_واو_diac.mp3", "pronunciation_style": "وَاْو (diacritic)", "cut_applied": False, "selection_notes": "Comparison winner"},
    {"label": "ي", "approved_filename": "letter_ي_yeh.mp3", "pronunciation_style": "standard (يِه)", "cut_applied": False, "selection_notes": "Default chosen mapping"},
]

APPROVED_NUMBER_EXPORT = [
    {"label": "0", "approved_filename": "num_00_v02.mp3", "pronunciation_style": "صِفِر v02", "cut_applied": False, "selection_notes": "Variant v02 preferred"},
    {"label": "1", "approved_filename": "num_01.mp3", "pronunciation_style": "وَاحِد", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "2", "approved_filename": "num_02_atneen_bang_cut_0_59.mp3", "pronunciation_style": "اتنين! + 0.59s cut", "cut_applied": True, "selection_notes": "Bang variant with leading-trim cut"},
    {"label": "3", "approved_filename": "num_03.mp3", "pronunciation_style": "تَلَاتَه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "4", "approved_filename": "num_04.mp3", "pronunciation_style": "أَرْبَعَه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "5", "approved_filename": "num_05.mp3", "pronunciation_style": "خَمْسَه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "6", "approved_filename": "num_06.mp3", "pronunciation_style": "سِتَّه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "7", "approved_filename": "num_07.mp3", "pronunciation_style": "سَبْعَه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "8", "approved_filename": "num_08.mp3", "pronunciation_style": "تَمَانْيَه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "9", "approved_filename": "num_09.mp3", "pronunciation_style": "تِسْعَه", "cut_applied": False, "selection_notes": "Standard"},
    {"label": "10", "approved_filename": "num_10_asharah_sukoon.mp3", "pronunciation_style": "عشرةْ (sukoon)", "cut_applied": False, "selection_notes": "Round-2 comparison winner"},
]

for row in APPROVED_LETTER_EXPORT:
    row["final_export_filename"] = f"letter_{row['label']}.mp3"
for row in APPROVED_NUMBER_EXPORT:
    row["final_export_filename"] = f"number_{row['label']}.mp3"

mapping_df = pd.DataFrame(APPROVED_LETTER_EXPORT + [
    {**r, "kind": "number"} for r in APPROVED_NUMBER_EXPORT
])
letter_map_df = pd.DataFrame(APPROVED_LETTER_EXPORT)
number_map_df = pd.DataFrame(APPROVED_NUMBER_EXPORT)

print("=== Letter export mapping ===")
display(letter_map_df[["label", "approved_filename", "pronunciation_style", "cut_applied", "final_export_filename", "selection_notes"]])

print("=== Number export mapping ===")
display(number_map_df[["label", "approved_filename", "pronunciation_style", "cut_applied", "final_export_filename", "selection_notes"]])


=== Letter export mapping ===


,label,approved_filename,pronunciation_style,cut_applied,final_export_filename,selection_notes
0,ا,letter_ا_alif.mp3,standard (أَلِف),False,letter_ا.mp3,Default chosen mapping
1,ب,letter_ب_cut_0.45.mp3,بِه + 0.45s duration cut,True,letter_ب.mp3,Trimmed beh clip; avoids long tail
2,ت,letter_ت_teh.mp3,standard (تِه),False,letter_ت.mp3,Default chosen mapping
3,ث,comp_seh_ث_ث_ه.mp3,ثِه (comp),False,letter_ث.mp3,Comparison winner
4,ج,letter_ج_geem.mp3,standard (جِيم),False,letter_ج.mp3,Default chosen mapping
5,ح,letter_ح_ha.mp3,standard (حَا),False,letter_ح.mp3,Default chosen mapping
6,خ,letter_خ_kha.mp3,standard (خَه),False,letter_خ.mp3,Default chosen mapping
7,د,comp_dal_د_دال_bang.mp3,دال! (bang),False,letter_د.mp3,Comparison winner over latin variant
8,ذ,comp_zal_ذ_ذال_bang.mp3,ذال! (bang),False,letter_ذ.mp3,Comparison winner
9,ر,letter_ر_ra.mp3,standard (رَا),False,letter_ر.mp3,Default chosen mapping


=== Number export mapping ===


,label,approved_filename,pronunciation_style,cut_applied,final_export_filename,selection_notes
0,0,num_00_v02.mp3,صِفِر v02,False,number_0.mp3,Variant v02 preferred
1,1,num_01.mp3,وَاحِد,False,number_1.mp3,Standard
2,2,num_02_atneen_bang_cut_0_59.mp3,اتنين! + 0.59s cut,True,number_2.mp3,Bang variant with leading-trim cut
3,3,num_03.mp3,تَلَاتَه,False,number_3.mp3,Standard
4,4,num_04.mp3,أَرْبَعَه,False,number_4.mp3,Standard
5,5,num_05.mp3,خَمْسَه,False,number_5.mp3,Standard
6,6,num_06.mp3,سِتَّه,False,number_6.mp3,Standard
7,7,num_07.mp3,سَبْعَه,False,number_7.mp3,Standard
8,8,num_08.mp3,تَمَانْيَه,False,number_8.mp3,Standard
9,9,num_09.mp3,تِسْعَه,False,number_9.mp3,Standard


## Export Final Audio

Copy **approved chosen** MP3s into the inference folders. Source files are never re-synthesized.

| Source | Destination |
|--------|-------------|
| `generated_audio/edge_tts_mapping_v2/chosen/` | `letters` |
| `generated_audio/edge_tts_numbers_mapping_v1/chosen/` | `numbers` |


In [6]:
def export_approved(chosen_dir: Path, export_dir: Path, records: list[dict], kind: str) -> list[dict]:
    logs = []
    for rec in records:
        src = chosen_dir / rec["approved_filename"]
        dest = export_dir / rec["final_export_filename"]
        if not src.is_file():
            raise FileNotFoundError(f"{kind} {rec['label']}: missing approved file {src}")
        shutil.copy2(src, dest)
        logs.append({**rec, "source_path": str(src.resolve()), "export_path": str(dest.resolve())})
        print(f"{kind} {rec['label']}: {rec['approved_filename']} -> {dest.name}")
    return logs

letter_export_log = export_approved(
    DIR_LETTERS_CHOSEN, DIR_EXPORT_LETTERS, APPROVED_LETTER_EXPORT, "letter"
)
number_export_log = export_approved(
    DIR_NUMBERS_CHOSEN, DIR_EXPORT_NUMBERS, APPROVED_NUMBER_EXPORT, "number"
)

print(f"\nExported {len(letter_export_log)} letters and {len(number_export_log)} numbers.")


letter ا: letter_ا_alif.mp3 -> letter_ا.mp3
letter ب: letter_ب_cut_0.45.mp3 -> letter_ب.mp3
letter ت: letter_ت_teh.mp3 -> letter_ت.mp3
letter ث: comp_seh_ث_ث_ه.mp3 -> letter_ث.mp3
letter ج: letter_ج_geem.mp3 -> letter_ج.mp3
letter ح: letter_ح_ha.mp3 -> letter_ح.mp3
letter خ: letter_خ_kha.mp3 -> letter_خ.mp3
letter د: comp_dal_د_دال_bang.mp3 -> letter_د.mp3
letter ذ: comp_zal_ذ_ذال_bang.mp3 -> letter_ذ.mp3


letter ر: letter_ر_ra.mp3 -> letter_ر.mp3
letter ز: comp_zay_ز_zeen_latin_long.mp3 -> letter_ز.mp3
letter س: comp_seen_س_seen_split_cut.mp3 -> letter_س.mp3
letter ش: letter_ش_sheen.mp3 -> letter_ش.mp3
letter ص: comp_sad_ص_saad_long_sukoon.mp3 -> letter_ص.mp3
letter ض: comp_dad_ض_daad_long_sukoon.mp3 -> letter_ض.mp3
letter ط: comp_ta_ط_ط_ه.mp3 -> letter_ط.mp3
letter ظ: letter_ظ_dha.mp3 -> letter_ظ.mp3
letter ع: letter_ع_ein.mp3 -> letter_ع.mp3
letter غ: letter_غ_ghein.mp3 -> letter_غ.mp3
letter ف: comp_faa_ف_ف_ه.mp3 -> letter_ف.mp3
letter ق: comp_qaf_ق_qaf_split_cut_v2.mp3 -> letter_ق.mp3


letter ك: comp_kaf_ك_كاف_sukoon.mp3 -> letter_ك.mp3


letter ل: comp_lam_ل_lam_latin.mp3 -> letter_ل.mp3
letter م: letter_م_meem.mp3 -> letter_م.mp3
letter ن: letter_ن_noon.mp3 -> letter_ن.mp3
letter ه: comp_ha_ه_ه_ه.mp3 -> letter_ه.mp3
letter و: comp_waw_و_واو_diac.mp3 -> letter_و.mp3
letter ي: letter_ي_yeh.mp3 -> letter_ي.mp3
number 0: num_00_v02.mp3 -> number_0.mp3
number 1: num_01.mp3 -> number_1.mp3
number 2: num_02_atneen_bang_cut_0_59.mp3 -> number_2.mp3
number 3: num_03.mp3 -> number_3.mp3
number 4: num_04.mp3 -> number_4.mp3
number 5: num_05.mp3 -> number_5.mp3
number 6: num_06.mp3 -> number_6.mp3
number 7: num_07.mp3 -> number_7.mp3
number 8: num_08.mp3 -> number_8.mp3
number 9: num_09.mp3 -> number_9.mp3
number 10: num_10_asharah_sukoon.mp3 -> number_10.mp3

Exported 28 letters and 11 numbers.


## Real-Time Pipeline Note

The gesture / prediction demo loads **`audio/letters/letter_<char>.mp3`** and **`audio/numbers/number_<n>.mp3`** at runtime.

Because Edge TTS can shift pronunciation between generations, we **do not** call Edge TTS during inference. The exported approved clips guarantee stable pronunciation in production.


## Optional: Archive Comparison Clips

Comparison MP3s are kept for documentation. Set `ARCHIVE_COMPARISON = True` to move non-chosen comparison files into `_archive/` (chosen and exported files are untouched).


In [7]:
ARCHIVE_COMPARISON = False

if ARCHIVE_COMPARISON:
    import time
    stamp = time.strftime("%Y%m%d")
    archive_root = GEN / "edge_tts_mapping_v2" / f"_archive_comparison_{stamp}"
    archive_root.mkdir(parents=True, exist_ok=True)
    moved = 0
    for folder in (DIR_LETTERS_COMP, DIR_LETTERS_COMP_CUT):
        if not folder.is_dir():
            continue
        dest_dir = archive_root / folder.relative_to(GEN / "edge_tts_mapping_v2")
        dest_dir.mkdir(parents=True, exist_ok=True)
        for mp3 in folder.glob("*.mp3"):
            chosen_copy = DIR_LETTERS_CHOSEN / mp3.name
            if chosen_copy.is_file():
                continue
            shutil.move(str(mp3), str(dest_dir / mp3.name))
            moved += 1
    print(f"Archived {moved} comparison MP3(s) under {archive_root}")
else:
    print("ARCHIVE_COMPARISON=False — comparison files left in place for demonstration.")


ARCHIVE_COMPARISON=False — comparison files left in place for demonstration.


## Verification

Confirm export counts and play sample clips from the final folders.


In [8]:
letter_exports = sorted(DIR_EXPORT_LETTERS.glob("letter_*.mp3"))
number_exports = sorted(DIR_EXPORT_NUMBERS.glob("number_*.mp3"))

print("Final letter exports:", len(letter_exports), f"(expected {len(APPROVED_LETTER_EXPORT)})")
print("Final number exports:", len(number_exports), f"(expected {len(APPROVED_NUMBER_EXPORT)})")

missing_letters = [r["label"] for r in APPROVED_LETTER_EXPORT if not (DIR_EXPORT_LETTERS / r["final_export_filename"]).is_file()]
missing_numbers = [r["label"] for r in APPROVED_NUMBER_EXPORT if not (DIR_EXPORT_NUMBERS / r["final_export_filename"]).is_file()]

assert not missing_letters, f"Missing letter exports: {missing_letters}"
assert not missing_numbers, f"Missing number exports: {missing_numbers}"
assert len(letter_exports) == len(APPROVED_LETTER_EXPORT)
assert len(number_exports) == len(APPROVED_NUMBER_EXPORT)

print("\nAll approved files exported successfully.")


Final letter exports: 28 (expected 28)
Final number exports: 11 (expected 11)

All approved files exported successfully.


In [9]:
SAMPLE_LETTERS = ["ا", "س", "ق", "ض"]
SAMPLE_NUMBERS = ["0", "2", "10"]

display(Markdown("### Sample letter playback (exported)"))
for ch in SAMPLE_LETTERS:
    path = DIR_EXPORT_LETTERS / f"letter_{ch}.mp3"
    rec = next(r for r in APPROVED_LETTER_EXPORT if r["label"] == ch)
    display(Markdown(f"**{ch}** — `{rec['approved_filename']}` → `{path.name}`"))
    display(Audio(filename=str(path)))

display(Markdown("### Sample number playback (exported)"))
for lab in SAMPLE_NUMBERS:
    path = DIR_EXPORT_NUMBERS / f"number_{lab}.mp3"
    rec = next(r for r in APPROVED_NUMBER_EXPORT if r["label"] == lab)
    display(Markdown(f"**{lab}** — `{rec['approved_filename']}` → `{path.name}`"))
    display(Audio(filename=str(path)))


### Sample letter playback (exported)

**ا** — `letter_ا_alif.mp3` → `letter_ا.mp3`

**س** — `comp_seen_س_seen_split_cut.mp3` → `letter_س.mp3`

**ق** — `comp_qaf_ق_qaf_split_cut_v2.mp3` → `letter_ق.mp3`

**ض** — `comp_dad_ض_daad_long_sukoon.mp3` → `letter_ض.mp3`

### Sample number playback (exported)

**0** — `num_00_v02.mp3` → `number_0.mp3`

**2** — `num_02_atneen_bang_cut_0_59.mp3` → `number_2.mp3`

**10** — `num_10_asharah_sukoon.mp3` → `number_10.mp3`